# B-30 Teacher 공통 fine-tuning
현재 Colab의 Drive에 저장한 B30_init.pth를 사용한다. 기존 B-0 모델은 수정하지 않는다.

- Colab에서 GPU 런타임을 선택하고 아래 셀을 위에서부터 실행한다.
- Git 저장소와 데이터로더 파일이 있으면 그대로 사용한다. 없을 때만 내려받는다.
- CUB 데이터가 있으면 재사용하고, 없으면 공식 배포 파일을 내려받는다.
- 기본 학습 설정은 기존 B-0와 같은 seed 0, 20 epoch, batch 8, AdamW, backbone lr 1e-5, head lr 1e-4, weight decay 0.01이다.
- 학습은 AMP float16, 검증은 FP32를 사용하며 테스트 정확도는 계산하지 않는다.
- 실행 기록은 비워 두었다. 실제 GPU 학습 결과는 Colab에서 확인한다.


## 1. Drive 연결, 경로와 학습 설정
현재 Drive 구조에 맞춘 기본 경로다. 초기 체크포인트는 CUB 학습 전 파일이어야 한다.
B0_init.pth도 같은 형식이면 input_path만 바꿔 사용할 수 있다.


In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo

import hashlib
import importlib.util
import json
import random
import subprocess
import sys
import time

import numpy as np
import torch
import torchvision
from torch import nn
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

drive.mount("/content/drive")
assert torch.cuda.is_available(), "Colab 런타임 유형에서 GPU를 선택하세요."
device = torch.device("cuda")

input_path = Path("/content/drive/MyDrive/ondeviceAI/initial_checkpoints/B30_from_B0_init.pth")
output_root = Path("/content/drive/MyDrive/ondeviceAI/checkpoints/B30_from_B0")
repo_dir = Path("/content/ondeviceAI")
data_dir = repo_dir / "data/CUB_200_2011/CUB_200_2011"

repo_url = "https://github.com/nicewei77-lang/ondeviceAI.git"
DATA_REF = "dev"

config = {
    "seed": 0,
    "epochs": 20,
    "batch_size": 8,
    "optimizer": "AdamW",
    "backbone_lr": 1e-5,
    "head_lr": 1e-4,
    "weight_decay": 0.01,
    "scheduler": None,
    "train_precision": "AMP float16",
    "validation_precision": "float32",
    "selection": "validation accuracy, then validation loss",
}

assert input_path.is_file(), f"초기 체크포인트 경로를 확인하세요: {input_path}"
print("GPU:", torch.cuda.get_device_name(0))
print("학습 시작 파일:", input_path)


Mounted at /content/drive
Mounted at /content/drive
GPU: Tesla T4
학습 시작 파일: /content/drive/MyDrive/ondeviceAI/initial_checkpoints/B30_init.pth
GPU: Tesla T4
학습 시작 파일: /content/drive/MyDrive/ondeviceAI/initial_checkpoints/B30_init.pth


## 2. 초기 체크포인트와 DINOv3 공식 코드
기존 DINOv3 저장소가 있으면 가져오기를 반복하지 않는다. 초기 파일에 기록된 코드 버전과
다르면 자동으로 기존 파일을 바꾸지 않고 중단한다. 코드가 없으면 clone 후 기록된 버전을 선택한다.


In [ ]:
init_checkpoint = torch.load(input_path, map_location="cpu", weights_only=True)

assert init_checkpoint["format_version"] == 1
assert init_checkpoint["checkpoint_type"] == "init", "학습 전 init 체크포인트를 사용하세요."
assert init_checkpoint["model_name"] == "dinov3_convnext_base"
assert init_checkpoint["num_classes"] == 200

dino_dir = Path("/content/dinov3")
expected_commit = init_checkpoint["dino_commit"]

if not dino_dir.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/facebookresearch/dinov3.git", str(dino_dir)],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(dino_dir), "checkout", expected_commit],
        check=True,
    )
else:
    print("기존 DINOv3 코드를 사용합니다:", dino_dir)

current_commit = subprocess.check_output(
    ["git", "-C", str(dino_dir), "rev-parse", "HEAD"], text=True,
).strip()

assert current_commit == expected_commit, (
    "기존 DINOv3 코드가 초기 파일에 기록된 버전과 다릅니다. "
    "기록된 버전의 코드 또는 새 런타임을 준비하세요."
)

if str(dino_dir) not in sys.path:
    sys.path.insert(0, str(dino_dir))

from dinov3.hub.backbones import dinov3_convnext_base

init_metadata = {
    key: value for key, value in init_checkpoint.items()
    if key != "model_state_dict"
}
pruning_spec = init_checkpoint["pruning_spec"]

print("DINOv3 코드 버전:", current_commit)
print("Pruning 구조 포함:", pruning_spec is not None)


DINOv3 코드 버전: 6876159a11b4df116f30f667f8c9888617df0751
Pruning 구조 포함: True
DINOv3 코드 버전: 6876159a11b4df116f30f667f8c9888617df0751
Pruning 구조 포함: True


## 3. 축소된 모델 구조와 가중치 복원
pretrained=False는 구조부터 생성하는 설정이다. 아래에서 저장된 초기 가중치를 직접 넣는다.
Pruning을 다시 계산하지 않고 저장된 내부 채널 수로 층을 만든다.


In [ ]:
teacher = dinov3_convnext_base(pretrained=False)
teacher.head = nn.Linear(teacher.embed_dim, init_checkpoint["num_classes"])

if pruning_spec is not None:
    expected_blocks = {
        f"stages.{stage_index}.{block_index}"
        for stage_index, stage in enumerate(teacher.stages)
        for block_index, block in enumerate(stage)
    }
    assert set(pruning_spec["blocks"]) == expected_blocks

    for name, info in pruning_spec["blocks"].items():
        block = teacher.get_submodule(name)
        assert block.pwconv1.out_features == info["hidden_before"]

        hidden_after = info["hidden_after"]
        indices = info["keep_indices"]
        assert hidden_after == len(indices)
        assert len(set(indices)) == hidden_after
        assert 0 < hidden_after <= info["hidden_before"]
        assert all(0 <= index < info["hidden_before"] for index in indices)

        block.pwconv1 = nn.Linear(
            block.pwconv1.in_features, hidden_after,
            bias=block.pwconv1.bias is not None,
        )
        block.pwconv2 = nn.Linear(
            hidden_after, block.pwconv2.out_features,
            bias=block.pwconv2.bias is not None,
        )

print(teacher.load_state_dict(init_checkpoint["model_state_dict"], strict=True))
teacher.requires_grad_(True)
teacher = teacher.to(device)
del init_checkpoint

print("전체 parameter:", f"{sum(p.numel() for p in teacher.parameters()):,}")
for index, stage in enumerate(teacher.stages):
    print(f"Stage {index} 내부 채널:", stage[0].pwconv1.out_features)


<All keys matched successfully>
전체 parameter: 62,633,144
Stage 0 내부 채널: 358
Stage 1 내부 채널: 717
Stage 2 내부 채널: 1434
Stage 3 내부 채널: 2867


## 4. 데이터로더 코드 준비
기존 저장소에 공통 모듈과 분할 파일이 있으면 그 파일을 사용한다. 없으면 GitHub에서
같은 커밋의 파일을 가져온다. 가져온 파일은 별도 실행 폴더에 보관하며 기존 코드를 덮어쓰지 않는다.

최신 데이터로더는 data_loader.py를 사용하므로 노트북만 내려받는 방식 대신 이 공통 모듈을
직접 불러온다. 소스 코드와 공유 분할 목록의 해시를 기록한다.


In [ ]:
required_loader_files = [
    "data_loader.py",
    "docs/splits/train_ids.txt",
    "docs/splits/validation_ids.txt",
    "docs/splits/test_ids.txt",
    "docs/splits/metadata.json",
]

if not (repo_dir / ".git").is_dir():
    if repo_dir.exists() and any(repo_dir.iterdir()):
        raise RuntimeError(
            f"Git 저장소가 아닌 기존 폴더가 있습니다: {repo_dir}. "
            "기존 파일을 보존하고 repo_dir와 data_dir를 확인하세요."
        )
    subprocess.run(
        ["git", "clone", "--branch", DATA_REF, repo_url, str(repo_dir)],
        check=True,
    )
else:
    print("기존 팀 저장소를 사용합니다:", repo_dir)

cache_note = Path("/content/ondeviceAI_loader_cache.json")
working_files_ready = all((repo_dir / name).is_file() for name in required_loader_files)

if working_files_ready:
    loader_commit = subprocess.check_output(
        ["git", "-C", str(repo_dir), "rev-parse", "HEAD"], text=True,
    ).strip()
    loader_origin = "existing_working_tree"
    loader_content = {
        name: (repo_dir / name).read_bytes() for name in required_loader_files
    }
elif cache_note.is_file():
    cached = json.loads(cache_note.read_text(encoding="utf-8"))
    cached_root = Path(cached["root"])
    if not all((cached_root / name).is_file() for name in required_loader_files):
        raise RuntimeError("데이터로더 캐시가 불완전합니다. 캐시 경로를 확인하세요.")
    loader_commit = cached["commit"]
    loader_origin = "existing_download_cache"
    loader_content = {
        name: (cached_root / name).read_bytes() for name in required_loader_files
    }
else:
    subprocess.run(
        ["git", "-C", str(repo_dir), "fetch", repo_url, DATA_REF], check=True,
    )
    loader_commit = subprocess.check_output(
        ["git", "-C", str(repo_dir), "rev-parse", "FETCH_HEAD"], text=True,
    ).strip()
    loader_origin = "downloaded_missing_files"
    loader_content = {
        name: subprocess.check_output(
            ["git", "-C", str(repo_dir), "show", f"{loader_commit}:{name}"]
        )
        for name in required_loader_files
    }

loader_file_hashes = {
    name: hashlib.sha256(content).hexdigest()
    for name, content in loader_content.items()
}
source_fingerprint = hashlib.sha256(
    json.dumps(loader_file_hashes, sort_keys=True).encode()
).hexdigest()

loader_snapshot = Path("/content/ondeviceAI_loader_runtime") / source_fingerprint[:16]
for name, content in loader_content.items():
    destination = loader_snapshot / name
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_bytes(content)

cache_note.write_text(
    json.dumps({"root": str(loader_snapshot), "commit": loader_commit}, indent=2),
    encoding="utf-8",
)

module_name = f"cub_data_{source_fingerprint[:16]}"
module_spec = importlib.util.spec_from_file_location(
    module_name, loader_snapshot / "data_loader.py",
)
cub = importlib.util.module_from_spec(module_spec)
sys.modules[module_name] = cub
module_spec.loader.exec_module(cub)

print("데이터로더 준비 방식:", loader_origin)
print("저장소 커밋:", loader_commit)
print("실제 파일 해시:", loader_file_hashes["data_loader.py"])


데이터로더 준비 방식: existing_working_tree
저장소 커밋: b8de137d50502d6d324f486db9c9d08e524ef707
실제 파일 해시: 792620d07b57bd5bed6f57fd84ddc3e844eb9ae9cad0c0728c16ab2f04749eb1
데이터로더 준비 방식: existing_working_tree
저장소 커밋: b8de137d50502d6d324f486db9c9d08e524ef707
실제 파일 해시: 792620d07b57bd5bed6f57fd84ddc3e844eb9ae9cad0c0728c16ab2f04749eb1


## 5. CUB 원본 데이터 준비
기존 이미지와 메타데이터가 있으면 다운로드하지 않는다. 데이터가 없으면 공식 배포 파일을
내려받고 MD5를 확인한 뒤 /content에 압축을 푼다. 다른 위치에 데이터가 있으면 첫 셀의
data_dir를 images.txt와 images 폴더가 들어 있는 위치로 바꾼다.

출처: [CaltechDATA CUB-200-2011](https://data.caltech.edu/records/65de6-vp158)


In [ ]:
metadata_files = [
    "images.txt", "image_class_labels.txt", "classes.txt", "train_test_split.txt",
]
data_ready = (
    (data_dir / "images").is_dir()
    and all((data_dir / name).is_file() for name in metadata_files)
)

if data_ready:
    print("기존 CUB 데이터를 사용합니다:", data_dir)
else:
    archive_path = Path("/content/CUB_200_2011.tgz")
    partial_path = Path("/content/CUB_200_2011.tgz.part")
    download_url = (
        "https://data.caltech.edu/records/65de6-vp158/"
        "files/CUB_200_2011.tgz?download=1"
    )

    if archive_path.is_file():
        downloaded_path = archive_path
        print("기존 CUB 압축파일을 사용합니다.")
    else:
        subprocess.run(
            ["wget", "-c", "-O", str(partial_path), download_url], check=True,
        )
        downloaded_path = partial_path

    with downloaded_path.open("rb") as file:
        checksum = hashlib.file_digest(file, "md5").hexdigest()
    assert checksum == "97eceeb196236b17998738112f37df78", (
        f"CUB 압축파일 검사 실패: {downloaded_path}. "
        "손상된 다운로드 파일을 정리한 뒤 다시 실행하세요."
    )

    if downloaded_path == partial_path:
        partial_path.replace(archive_path)

    data_dir.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["tar", "-xzf", str(archive_path), "-C", str(data_dir.parent)],
        check=True,
    )

assert (data_dir / "images").is_dir()
assert all((data_dir / name).is_file() for name in metadata_files)
print("데이터 준비 완료:", data_dir)


데이터 준비 완료: /content/ondeviceAI/data/CUB_200_2011/CUB_200_2011


## 6. 데이터 분할과 전처리 검사
공통 모듈로 분할과 transform을 구성한다. 공유 ID 목록과 순서를 대조하고, 중복과 실제 이미지
누락을 검사한다. 테스트 집합은 ID만 확인하며 평가에 사용하지 않는다.


In [ ]:
image_list, image_labels, class_names = cub.read_cub_metadata(data_dir)
train_image_list, val_image_list, test_image_list = cub.split_cub_images(
    data_dir, image_list, image_labels,
)
train_transform, eval_transform = cub.build_transforms()

assert len(image_list) == 11788
assert set(image_labels.values()) == set(range(200))
assert set(class_names) == set(range(200))

source_metadata = json.loads(
    (loader_snapshot / "docs/splits/metadata.json").read_text(encoding="utf-8")
)
source_metadata_hashes = {}
for name in metadata_files:
    actual_hash = hashlib.sha256((data_dir / name).read_bytes()).hexdigest()
    source_metadata_hashes[name] = actual_hash
    expected_hash = source_metadata.get("source_sha256", {}).get(name)
    if expected_hash is not None:
        assert actual_hash == expected_hash, f"원본 메타데이터 불일치: {name}"

missing_images = [
    row.split(maxsplit=1)[1]
    for row in image_list
    if not (data_dir / "images" / row.split(maxsplit=1)[1]).is_file()
]
assert not missing_images, f"이미지 파일 누락: {len(missing_images)}장"

expected_counts = {"train": 4794, "validation": 1200, "test": 5794}
split_id_sets = {}
for split_name, rows in [
    ("train", train_image_list),
    ("validation", val_image_list),
    ("test", test_image_list),
]:
    actual_ids = [row.split()[0] for row in rows]
    reference_ids = (
        loader_snapshot / "docs/splits" / f"{split_name}_ids.txt"
    ).read_text().split()
    assert actual_ids == reference_ids, f"{split_name} 분할 순서 불일치"
    assert len(actual_ids) == expected_counts[split_name]
    assert len(set(actual_ids)) == len(actual_ids), f"{split_name} 중복 ID"
    split_id_sets[split_name] = set(actual_ids)
    print(f"{split_name}: {len(actual_ids)}장, ID와 순서 일치")

assert split_id_sets["train"].isdisjoint(split_id_sets["validation"])
assert split_id_sets["train"].isdisjoint(split_id_sets["test"])
assert split_id_sets["validation"].isdisjoint(split_id_sets["test"])

train_dataset = cub.BirdDataset(
    train_image_list, data_dir, image_labels, train_transform,
)
val_dataset = cub.BirdDataset(
    val_image_list, data_dir, image_labels, eval_transform,
)

print("학습 전처리:", train_transform)
print("검증 전처리:", eval_transform)


train: 4794장, ID와 순서 일치
validation: 1200장, ID와 순서 일치
test: 5794장, ID와 순서 일치
학습 전처리: Compose(
    RandomHorizontalFlip(p=0.5)
    ColorJitter(brightness=(0.9, 1.1), contrast=None, saturation=None, hue=None)
    <function pad_image at 0x7c9548377b00>
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)
검증 전처리: Compose(
    <function pad_image at 0x7c9548377b00>
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)


## 7. 학습 배치와 새 optimizer
복원한 모델에 대해 optimizer를 새로 생성한다. 학습 순서와 증강 난수의 시작점을 고정한다.
같은 seed라도 GPU와 라이브러리가 다르면 결과가 완전히 일치하는 것은 보장되지 않는다.


In [ ]:
random.seed(config["seed"])
np.random.seed(config["seed"])
torch.manual_seed(config["seed"])
torch.cuda.manual_seed_all(config["seed"])

train_generator = torch.Generator()
train_generator.manual_seed(config["seed"])

teacher_train_loader = DataLoader(
    train_dataset, batch_size=config["batch_size"], shuffle=True,
    num_workers=0, pin_memory=True, generator=train_generator,
)
teacher_val_loader = DataLoader(
    val_dataset, batch_size=config["batch_size"], shuffle=False,
    num_workers=0, pin_memory=True,
)

backbone_parameters = [
    parameter for name, parameter in teacher.named_parameters()
    if not name.startswith("head.")
]
optimizer = torch.optim.AdamW(
    [
        {"params": backbone_parameters, "lr": config["backbone_lr"]},
        {"params": teacher.head.parameters(), "lr": config["head_lr"]},
    ],
    weight_decay=config["weight_decay"],
)
criterion = nn.CrossEntropyLoss()
scaler = torch.amp.GradScaler("cuda")

print("학습 이미지:", len(train_dataset))
print("검증 이미지:", len(val_dataset))
print("학습 배치:", len(teacher_train_loader))
print("검증 배치:", len(teacher_val_loader))


학습 이미지: 4794
검증 이미지: 1200
학습 배치: 600
검증 배치: 150
학습 이미지: 4794
검증 이미지: 1200
학습 배치: 600
검증 배치: 150


## 8. 한 epoch 함수
학습은 AMP float16, 검증은 FP32다. 마지막 작은 배치를 포함해 이미지 수 기준으로 평균을 계산한다.


In [ ]:
def run_epoch(loader, training):
    teacher.train(training)
    total_loss = 0.0
    total_correct = 0
    total_images = 0

    progress = tqdm(
        loader, desc="Train" if training else "Validation", leave=False,
    )
    for images, labels in progress:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):
            with torch.autocast(
                device_type="cuda", dtype=torch.float16, enabled=training,
            ):
                logits = teacher(images)
                loss = criterion(logits, labels)

            if not torch.isfinite(loss).item():
                raise RuntimeError("Loss에 NaN 또는 Inf가 발생했습니다.")

            if training:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_correct += (
            logits.detach().argmax(dim=1) == labels
        ).sum().item()
        total_images += batch_size

    if training:
        optimizer.zero_grad(set_to_none=True)

    return total_loss / total_images, total_correct / total_images


## 9. 결과 폴더와 재현 정보
매 실행마다 새 Drive 폴더를 생성하므로 기존 B-0와 결과가 섞이지 않는다.
입력 체크포인트, 사용한 데이터로더 파일과 분할 목록의 해시를 기록한다.


In [ ]:
if pruning_spec is None:
    condition = "B0"
else:
    condition = f"B{round(100 * pruning_spec['requested_ratio'])}"

run_name = datetime.now(ZoneInfo("Asia/Seoul")).strftime(
    f"{condition}_%Y%m%d_%H%M%S_%f"
)
run_dir = output_root / run_name
run_dir.mkdir(parents=True, exist_ok=False)
best_path = run_dir / "best.pth"

with input_path.open("rb") as file:
    input_checkpoint_sha256 = hashlib.file_digest(file, "sha256").hexdigest()

config.update({
    "condition": condition,
    "input_checkpoint": str(input_path),
    "input_checkpoint_sha256": input_checkpoint_sha256,
    "model": init_metadata["model_name"],
    "dino_commit": current_commit,
    "input_size": 224,
    "train_transform": repr(train_transform),
    "eval_transform": repr(eval_transform),
    "split_counts": expected_counts,
    "source_metadata_sha256": source_metadata_hashes,
    "data_directory": str(data_dir),
    "torch_version": str(torch.__version__),
    "torchvision_version": str(torchvision.__version__),
    "cuda_version": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0),
    "data_loader_repository": repo_url,
    "data_loader_commit": loader_commit,
    "data_loader_origin": loader_origin,
    "data_loader_file_sha256": loader_file_hashes,
})

(run_dir / "config.json").write_text(
    json.dumps(config, ensure_ascii=False, indent=2), encoding="utf-8",
)
if pruning_spec is not None:
    (run_dir / "pruning_spec.json").write_text(
        json.dumps(pruning_spec, ensure_ascii=False, indent=2), encoding="utf-8",
    )
print("이번 학습 저장 위치:", run_dir)


이번 학습 저장 위치: /content/drive/MyDrive/ondeviceAI/checkpoints/B30_20261026_143002_691983


## 10. 학습 시작
이 셀부터 실제로 가중치를 갱신한다. 최고 검증 정확도의 모델을 선택하며, 동률이면 낮은 검증
loss를 선택한다. Pruning 구조도 best.pth에 함께 저장한다.

best.pth는 평가와 KD에 사용할 모델 복원 파일이다. optimizer와 scaler 상태는 포함하지 않으므로
중단 시점의 학습을 그대로 이어가는 체크포인트는 아니다.


In [ ]:
best_score = (-1.0, float("-inf"))
history = []

for epoch in range(1, config["epochs"] + 1):
    start_time = time.perf_counter()
    train_loss, train_acc = run_epoch(teacher_train_loader, training=True)
    val_loss, val_acc = run_epoch(teacher_val_loader, training=False)
    elapsed = time.perf_counter() - start_time

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_accuracy": train_acc,
        "validation_loss": val_loss,
        "validation_accuracy": val_acc,
        "seconds": elapsed,
    })
    print(
        f"Epoch {epoch:02d}/{config['epochs']} | "
        f"Train loss {train_loss:.4f}, acc {train_acc:.2%} | "
        f"Val loss {val_loss:.4f}, acc {val_acc:.2%} | "
        f"{elapsed:.1f}초"
    )

    score = (val_acc, -val_loss)
    if score > best_score:
        checkpoint = {
            **init_metadata,
            "checkpoint_type": "finetuned",
            "model_state_dict": {
                name: tensor.detach().cpu().clone()
                for name, tensor in teacher.state_dict().items()
            },
            "epoch": epoch,
            "validation_accuracy": val_acc,
            "validation_loss": val_loss,
            "config": config,
            "class_names": class_names,
        }
        temporary_path = run_dir / "best.tmp.pth"
        torch.save(checkpoint, temporary_path)
        temporary_path.replace(best_path)
        best_score = score
        del checkpoint
        print("  → 최고 검증 모델 저장 완료")

    (run_dir / "history.json").write_text(
        json.dumps(history, ensure_ascii=False, indent=2), encoding="utf-8",
    )

print("학습 종료")
print("최고 모델:", best_path)


Train:   0%|          | 0/600 [00:00<?, ?it/s]

Train:   0%|          | 0/600 [00:00<?, ?it/s]

Validation:   0%|          | 0/150 [00:00<?, ?it/s]

Validation:   0%|          | 0/150 [00:00<?, ?it/s]

Epoch 01/20 | Train loss 5.3455, acc 0.60% | Val loss 5.1884, acc 0.92% | 149.0초
Epoch 01/20 | Train loss 5.3455, acc 0.60% | Val loss 5.1884, acc 0.92% | 149.0초
  → 최고 검증 모델 저장 완료
  → 최고 검증 모델 저장 완료


...

  → 최고 검증 모델 저장 완료


Train:   0%|          | 0/600 [00:00<?, ?it/s]

Validation:   0%|          | 0/150 [00:00<?, ?it/s]

Validation:   0%|          | 0/150 [00:00<?, ?it/s]

Epoch 20/20 | Train loss 0.0889, acc 97.52% | Val loss 2.9984, acc 44.50% | 146.8초
학습 종료
최고 모델: /content/drive/MyDrive/ondeviceAI/checkpoints/B30_20261026_143002_691983/best.pth
Epoch 20/20 | Train loss 0.0889, acc 97.52% | Val loss 2.9984, acc 44.50% | 146.8초
학습 종료
최고 모델: /content/drive/MyDrive/ondeviceAI/checkpoints/B30_20261026_143002_691983/best.pth


## 11. 최고 모델 복원
마지막 epoch 대신 선택된 최고 모델을 복원한다. 같은 검증 집합으로 결과를 재확인한다.
테스트 집합은 사용하지 않는다.


In [ ]:
best_checkpoint = torch.load(best_path, map_location="cpu", weights_only=True)
teacher.load_state_dict(best_checkpoint["model_state_dict"], strict=True)
teacher.eval()

stored_accuracy = best_checkpoint["validation_accuracy"]
print("선택된 epoch:", best_checkpoint["epoch"])
print("최고 검증 정확도:", f"{stored_accuracy:.2%}")
print("해당 검증 loss:", f"{best_checkpoint['validation_loss']:.4f}")
del best_checkpoint

restored_loss, restored_accuracy = run_epoch(
    teacher_val_loader, training=False,
)
print("복원 후 검증 정확도:", f"{restored_accuracy:.2%}")
print("복원 후 검증 loss:", f"{restored_loss:.4f}")
assert abs(restored_accuracy - stored_accuracy) < 1e-12, (
    "복원 전후 검증 정확도가 다릅니다."
)


선택된 epoch: 19
최고 검증 정확도: 45.33%
해당 검증 loss: 2.9838


Validation:   0%|          | 0/150 [00:00<?, ?it/s]

복원 후 검증 정확도: 45.33%
복원 후 검증 loss: 2.9838


## 저장 파일과 공유
- best.pth: 학습된 Teacher, 클래스 정보와 pruning 구조
- config.json: 경로, 학습 설정, 사용한 소스 파일 해시와 실행 환경
- history.json: epoch별 학습과 검증 기록
- pruning_spec.json: 축소된 구조 기록

가중치는 Drive로 공유하고 GitHub에는 노트북과 필요한 실험 기록을 올린다.
KD 담당자는 pruning_spec대로 모델 구조를 구성한 뒤 model_state_dict를 복원한다.


In [ ]:
b0_path = Path(
    "/content/drive/MyDrive/ondeviceAI/checkpoints/"
    "B0_20261024_172759_400529/best.pth"
)

assert b0_path.is_file(), f"B-0 체크포인트를 찾을 수 없습니다: {b0_path}"

b0_checkpoint = torch.load(
    b0_path,
    map_location="cpu",
    weights_only=True,
)

# 원본 크기의 모델과 200클래스 분류층을 생성한다.
b0_model = dinov3_convnext_base(pretrained=False)
b0_model.head = nn.Linear(b0_model.embed_dim, 200)

print(
    b0_model.load_state_dict(
        b0_checkpoint["model_state_dict"],
        strict=True,
    )
)

# 현재 데이터로더와 클래스 번호의 의미가 같은지 확인한다.
assert b0_checkpoint["class_names"] == class_names

b0_saved_accuracy = b0_checkpoint["validation_accuracy"]

b0_metadata = {
    key: value
    for key, value in b0_checkpoint.items()
    if key != "model_state_dict"
}

del b0_checkpoint

b0_model = b0_model.to(device)
b0_model.eval()

print("B-0 저장 당시 검증 정확도:", f"{b0_saved_accuracy:.2%}")

<All keys matched successfully>
B-0 저장 당시 검증 정확도: 83.33%


In [ ]:
@torch.no_grad()
def evaluate_teacher(model, loader):
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_images = 0

    for images, labels in tqdm(loader, desc="B-0 validation"):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        # 기존 검증과 동일하게 FP32로 평가한다.
        with torch.autocast(device_type="cuda", enabled=False):
            logits = model(images)
            loss = criterion(logits, labels)

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total_images += batch_size

    return total_loss / total_images, total_correct / total_images


b0_val_loss, b0_val_accuracy = evaluate_teacher(
    b0_model,
    teacher_val_loader,
)

print("저장 당시 정확도:", f"{b0_saved_accuracy:.2%}")
print("현재 검증 정확도:", f"{b0_val_accuracy:.2%}")
print("현재 검증 loss:", f"{b0_val_loss:.4f}")

# 환경 차이를 고려해 검증 이미지 1장 이내의 정확도 차이를 허용한다.
tolerance = 1 / len(teacher_val_loader.dataset) + 1e-12

assert abs(b0_val_accuracy - b0_saved_accuracy) <= tolerance, (
    "B-0 검증 결과가 이전과 다릅니다. "
    "Pruning 전에 데이터, 전처리와 모델 복원을 확인해야 합니다."
)

print("B-0 재평가 확인 완료")


B-0 validation:   0%|          | 0/150 [00:00<?, ?it/s]

저장 당시 정확도: 83.33%
현재 검증 정확도: 83.33%
현재 검증 loss: 0.7331
B-0 재평가 확인 완료
